This is a simple Data Question-Answer (QA) bot that can fetch data from the relevant tables/files <br>
based on the user query by running a python query which is relevant to the question. <br>
Note : this is a bot without memory

Pre-requisites : Mistral API Key

In [25]:
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_core.tools import Tool, tool
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from typing import Annotated, TypedDict, Sequence
from langgraph.prebuilt import create_react_agent
from langchain_experimental.utilities import PythonREPL
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages.base import BaseMessage
import operator

In [26]:
import uuid

thread_id = uuid.uuid4()
config = {"configurable":{"thread_id":thread_id}}
memory = MemorySaver()

In [ ]:
import os
os.environ['MISTRAL_API_KEY'] = ''
llm = ChatMistralAI(model_name='mistral-large-latest') 

In [28]:
class State(TypedDict):
    messages : Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps : Annotated[Sequence[BaseMessage], "the remaining steps"]

In [88]:
@tool
def execute_python_code(py_code : Annotated[str, 'The python code to execute'])->str:
    """Executes the input python code as it and returns the result"""
    python_repl = PythonREPL()
    py_code = py_code
    return python_repl.run(py_code)
tools = [execute_python_code]

In [89]:
system_prompt = """You are a helpful assistant. You answer user queries by using the context mentioned below 
and by writing python functions and calling the required tools to execute them.

<context>
Available csv files to fetch results based on the user query - [products.csv]. All column names are are in lowercase.
</context>

If you still do not find the answer simple reply as - ```Sorry I am unable to help```

Your thought : {{agent_scratchpad}}
"""

In [90]:
agent_executor = create_react_agent(llm, prompt=system_prompt, tools=tools)

In [ ]:
input_message = [{"role":"user",
    "content":"what are the available products"}]

for i, step in enumerate(agent_executor.stream(
    {"messages": input_message}, stream_mode="values"
)):
    step["messages"][-1].pretty_print()
    ## to interrupt and ask consent before executing python code (safety feature) 
    if 'tool_calls' in step["messages"][i].additional_kwargs.keys():
        consent = input("Do you wish to process? (y/n)")
        if 'n' in consent.lower():
            break


================================ Human Message =================================

what are the available products
================================== Ai Message ==================================
Tool Calls:
  execute_python_code (X8Wy9cFc9)
 Call ID: X8Wy9cFc9
  Args:
    py_code: import pandas as pd

# Load the CSV file
products_df = pd.read_csv('products.csv')

# Get the unique product names
product_names = products_df['product_name'].unique()

# Print the product names
for product in product_names:
    print(product)


Python REPL can execute arbitrary code. Use with caution.


================================= Tool Message =================================
Name: execute_python_code

lakme red lipstick
lakme nude lipstick
lakme pink lipstick
lakme brown lipstick
huda red lipstick
huda nude lipstick
huda pink lipstick
huda brown lipstick
huda concealor-1
huda concealor-2
maybelline concealor-1
maybelline concealor-2

================================== Ai Message ==================================

Here are the available products

1. Lakme red lipstick
2. Lakme nude lipstick
3. Lakme pink lipstick
4. Lakme brown lipstick
5. Huda red lipstick
6. Huda nude lipstick
7. Huda pink lipstick
8. Huda brown lipstick
9. Huda concealor-1
10. Huda concealor-2
11. Maybelline concealor-1
12. Maybelline concealor-2
